In [365]:
import os
from tqdm.auto import tqdm
import pandas as pd

# to use data structures
from pydantic import BaseModel, ConfigDict
# from enum import Enum

# OpenAI API
from openai import OpenAI

# Load the API keys from .env
from dotenv import load_dotenv

pd.options.mode.chained_assignment = (
    None  # default='warn' This disables warning of "copying a slice of a DataFrame"
)
tqdm.pandas()  # activate the tqdm for pandas

In [366]:
# load the API keys
load_dotenv(".env")
for api_key in ["OPENAI_API_KEY"]:
    if os.getenv(api_key) is not None:
        print(api_key, "loaded")
    else:
        print(api_key, "missing")
        print("Please create a .env file with the corresponding API key")

OPENAI_API_KEY loaded


In [367]:
from utils.set_paths import (
    path_app_context,
    path_templates,
    path_data,
    path_output,
)

path_batch_files = path_data / "batches"

# import util functions
from utils.content_utils import *
from utils.file_ops import *

In [368]:
summary_excel = pd.read_excel("../EconJM_status.xlsx", sheet_name="sum")
summary_excel.tail(5)

,add_id,app_notes,good_match,app_status,outcome,network email,institution,location,add_rank,add_field,...,add_posting,add_how_apply,other_link,how_to_rec_letters,need_cover_letter,need_RS,need_TS,OMER_letter_status,KLAUS_letter_status,WOO_letter_status
94,oslo,NaN,NaN,generating docs,NaN,NaN,University of Oslo,NaN,assistant,NaN,...,EJM,NaN,NaN,NaN,1,1,1,NaN,NaN,NaN
95,usnw,NaN,NaN,generating docs,NaN,NaN,University of New South Wales,NaN,assistant,NaN,...,EJM,NaN,NaN,NaN,1,0,0,NaN,NaN,NaN
96,alicante,NaN,NaN,generating docs,NaN,NaN,Universidad de Alicante,NaN,assistant,macroeconomics,...,EJM,NaN,NaN,NaN,1,0,0,NaN,NaN,NaN
97,cyberagent,NaN,NaN,generating docs,NaN,NaN,"CyberAgent, Inc.",NaN,NaN,NaN,...,JOE,NaN,NaN,NaN,1,0,0,NaN,NaN,NaN
98,columbia,NaN,NaN,completed,NaN,NaN,Columbia University,NaN,assistant,NaN,...,EJM,https://apply.interfolio.com/175578,NaN,NaN,0,0,0,NaN,NaN,NaN


# Load the needed input files

- Research Statement Template
- Cover Letter Template
- Teaching Statement Template

In [369]:
# Research Statement template
with open(path_templates / "base_text/research_statement.txt", "r") as f:
    lines = f.readlines()

RA_TEMPLATE = " ".join(lines)

with open(path_templates / "base_text/teaching_statement.txt", "r") as f:
    lines = f.readlines()

TA_TEMPLATE = " ".join(lines)

with open(path_templates / "base_text/cover_letter.txt", "r") as f:
    lines = f.readlines()

CL_TEMPLATE = " ".join(lines)

# Figure out what docs are needed to generate

In [370]:
columns_to_show = [
    "add_id",
    "institution",
    "location",
    "add_rank",
    "add_field",
    "add_category",
    "need_cover_letter",
    "need_RS",
    "need_TS",
]

In [371]:
completed_docs = [str(d).split("/")[-1] for d in path_output.iterdir() if d.is_dir()]

docs_to_generate = set(summary_excel.add_id.to_list()) - set(completed_docs)

if docs_to_generate:
    print("Need to generate docs for:")
    to_generate_df = summary_excel[summary_excel.add_id.isin(docs_to_generate)][
        columns_to_show
    ]
    docs_list = to_generate_df.astype(str).values.tolist()
    for id in docs_to_generate:
        print(f"- {id}")
else:
    print("No docs needed to generate")

Need to generate docs for:
- hgse
- toronto
- vancouver
- bi
- lse
- toronto_2
- usnw
- columbia
- manchester
- alicante
- bank_port
- bank_slovakia
- oslo
- cafoscari
- hanken


In [372]:
to_generate_df.head()

,add_id,institution,location,add_rank,add_field,add_category,need_cover_letter,need_RS,need_TS
42,toronto,University of Toronto,NaN,lecturer,NaN,teaching,1,0,1
65,toronto_2,University of Toronto,NaN,assistant,NaN,research,1,0,1
66,vancouver,University of British Columbia,NaN,assistant,NaN,research,1,0,1
67,cafoscari,Ca' Foscari University Venice,NaN,postdoc,NaN,research,1,0,0
68,manchester,University of Manchester,NaN,assistant,NaN,research,1,0,0


# Add the context and additional data for each prompt

For these all the `raw` documents are transcribed in a few bullet points:

- For Job add use 
    ```
    summarize the job add in a few bullet points. Prioritize the attributes, qualitites they look for in a candidate
    ```
- For the isntitutional values use 
    ```
    summarize the institution mission and values in a few bullet points
    ```
- For econ research department use 
    ```
    summarize the economics department or center values and research topics in a few bullet points. Make emphasis on potential co-authors and researcg/teaching groups described there and write it in the file
    ```

In [373]:
# get the add description context
to_generate_df.loc[:, "add_description"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START JOB DESCRIPTION",
        end_marker="END JOB DESCRIPTION",
    ),
    axis=1,
)

to_generate_df.loc[:, "econ_context"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START ECON DEPT",
        end_marker="END ECON DEPT",
    ),
    axis=1,
)

to_generate_df.loc[:, "institution_values"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START INSTITUTIONAL DESCRIPTION",
        end_marker="END INSTITUTIONAL DESCRIPTION",
    ),
    axis=1,
)

# OpenAI Batches

In [374]:
openai_client = OpenAI()
model_name = "gpt-4.1-mini"

In [375]:
# Load the OpenAI batching df
# load history of batches
if (path_batch_files / "batches_history.csv").exists():
    batch_history = pd.read_csv(path_batch_files / "batches_history.csv")
else:
    print("No history of batches found")
    batch_history = pd.DataFrame(
        columns=[
            "created_at",
            "description",
            "model",
            "cat_type",
            "batch_id",
            "status",
            "processing_status",
            "input_file_id",
            "output_file_id",
            "error_file_id",
        ]
    )

batch_history = update_batch_status(batch_history, openai_client)
batch_history.sort_values(["created_at", "model", "cat_type"], inplace=True)
batch_history.to_csv(path_batch_files / "batches_history.csv", index=False)


# Check the inprocess batches
in_process_batches = batch_history[batch_history["status"] == "in_progress"]
print("In-process batches:")
display(in_process_batches.tail(5))

# Check the completed batches
completed_batches = batch_history[
    (batch_history["status"] == "completed")
    & (batch_history["processing_status"] == False)
    & (batch_history["error_file_id"].isna())
]
print("Completed batches:")
display(completed_batches.tail(5))

In-process batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id


Completed batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id
41,2025-10-19 17:48:39.086997,Generate Cover Letters,gpt-4.1-mini,cl,batch_68f56ac7c9f08190ae069432c6b96f12,completed,False,file-Ti8tVZRCSJERq75gdLjdtK,file-CfPHnL5eW4d2hwZFFK8Kb9,NaN
42,2025-10-19 17:48:46.162565,Generate Teaching Statements,gpt-4.1-mini,ts,batch_68f56aceee388190a9038cf1287a2598,completed,False,file-HotkgbwCuBGDi7K9w25QME,file-GCVX7gXpwJLmv36E9PKBuV,NaN


In [376]:
# batch_history

In [377]:
class BodyText(BaseModel):
    body_text: str
    cot: str
    model_config = ConfigDict(extra="forbid")

# Cover Letter

In [378]:
gen_aux = to_generate_df[to_generate_df.need_cover_letter == 1].copy()
# gen_aux

In [379]:
def cl_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to add a position fit paragraph to the following COVER LETTER START and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    Keep the length of the paragraph to about 100 words.
    Use a professional and academic tone, but make it engaging and easy to read.
    For each statement of fit add an EXAMPLE from my research, teaching, or service experience that demostrates the fit statement.
    For example, for interdisciplinary research you can add that I won Dedman College Interdisciplinary Institute’s Inaugural Graduate Student Summer Research and Writing Fellowship.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full new paragraph as `body_text` and a short chain-of-thought explanation of how you modified the BASE COVER LETTER START to fit the institution and department as `cot`.
    The text should follow the Typst formatting in the BASE COVER LETTER START.

    BASE COVER LETTER START:
    {CL_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [380]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: cl_prompt(row),
    axis=1,
)

In [381]:
name_jsonl = "gen_cl"
description_batch = "Generate Cover Letters"

content_schema = BodyText
cat_type = "cl"

In [382]:
%run -i batched_chat_gpt.py

Found completed batches for gpt-4.1-mini and CL
Updating the existing old_docs
Need to generate 11 CL docs that contain a using gpt-4.1-mini
Creating a new batch
Created a new batch with id batch_68f6f0f67b748190b105f3dc90c6e68c and updated the batch history.


# Research Statement

In [383]:
gen_aux = to_generate_df[to_generate_df.need_RS == 1].copy()
# gen_aux

In [384]:
def rs_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to modify the following BASE RESEARCH STATEMENT and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    In addition, highlight my research fit to the JOB ADVERTISEMENT qualities described.
    Keep the length of the research statement to about 600 words and keep the new content as brief as possible.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full research statement as `body_text` and a short chain-of-thought explanation of how you modified the BASE RESEARCH STATEMENT to fit the institution and department as `cot`.
    The text should follow the Typst format.

    BASE RESEARCH STATEMENT:
    {RA_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [385]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: rs_prompt(row),
    axis=1,
)

In [386]:
name_jsonl = "gen_rs"
description_batch = "Generate Research Statements"

content_schema = BodyText
cat_type = "rs"

In [387]:
%run -i batched_chat_gpt.py

Need to generate 3 RS docs that contain a using gpt-4.1-mini
Creating a new batch
Created a new batch with id batch_68f6f0f7c51c8190bb59990bef192fe7 and updated the batch history.


# Teaching Statement

In [388]:
gen_aux = to_generate_df[to_generate_df.need_TS == 1].copy()
# gen_aux

In [389]:
def ts_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to modify the following BASE TEACHING STATEMENT and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    In addition, highlight my educational philosophy fit to the JOB ADVERTISEMENT qualities described.
    Keep the length of the teaching statement to about 300 words and keep the new content as brief as possible.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full teaching statement as `body_text` and a short chain-of-thought explanation of how you modified the BASE TEACHING STATEMENT to fit the institution and department as `cot`.
    The text should follow the Typst format.

    BASE TEACHING STATEMENT:
    {TA_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [390]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: ts_prompt(row),
    axis=1,
)

In [391]:
name_jsonl = "gen_ts"
description_batch = "Generate Teaching Statements"

content_schema = BodyText
cat_type = "ts"

In [392]:
%run -i batched_chat_gpt.py

Found completed batches for gpt-4.1-mini and TS
Updating the existing old_docs
Need to generate 4 TS docs that contain a using gpt-4.1-mini
Creating a new batch
Created a new batch with id batch_68f6f0f8cd908190b09b53a230b5e44b and updated the batch history.


# Generate the Docs

Here we assume that all docs have been generated

In [393]:
def generate_docs(df, generated_df, gen_type="rs"):
    if gen_type == "rs":
        dir_name = "research_statement"
        file_name = "Gonzalez_RS.typ"
    elif gen_type == "ts":
        dir_name = "teaching_statement"
        file_name = "Gonzalez_TS.typ"
    elif gen_type == "cl":
        dir_name = "cover_letter"
        file_name = "Gonzalez_CL.typ"
    else:
        raise ValueError("gen_type not recognized")

    for i, row in df.iterrows():
        print(f"Processing {row.add_id}...")

        new_content = generated_df[
            generated_df.add_id == row.add_id
        ].body_text_1.values[0]

        target_path = path_output / row.add_id
        subdir_path = target_path / dir_name

        if not target_path.exists():
            print(f"Creating directory: {target_path}")
            copy_and_rename_dir(path_templates / f"{dir_name}", target_path, dir_name)
        else:
            print(f"Main Directory already exists: {target_path}")
            if not subdir_path.exists():
                print(f"Creating subdirectory: {subdir_path}")
                copy_and_rename_dir(
                    path_templates / f"{dir_name}", target_path, dir_name
                )

        try:
            add_lines_to_typs_doc(subdir_path / file_name, new_content)
            print(f"{file_name} added for {row.add_id}")
        except Exception as e:
            print(f"Error processing {row.add_id}: {e}")

## Cover Letter

In [394]:
cl_gen_df = pd.read_csv(path_output / "cl_docs_gpt-4.1-mini.csv")

gen_aux = to_generate_df[to_generate_df.need_cover_letter == 1].copy()

generate_docs(gen_aux, cl_gen_df, gen_type="cl")

Processing toronto...
Creating directory: ../output_docs/toronto
Gonzalez_CL.typ added for toronto
Processing toronto_2...
Creating directory: ../output_docs/toronto_2
Gonzalez_CL.typ added for toronto_2
Processing vancouver...
Creating directory: ../output_docs/vancouver
Gonzalez_CL.typ added for vancouver
Processing cafoscari...


IndexError: index 0 is out of bounds for axis 0 with size 0

## Research Statement

In [ ]:
rs_gen_df = pd.read_csv(path_output / "rs_docs_gpt-4.1-mini.csv")

gen_aux = to_generate_df[to_generate_df.need_RS == 1].copy()

generate_docs(gen_aux, rs_gen_df, gen_type="rs")

Processing stavanger...
Main Directory already exists: ../output_docs/stavanger
Creating subdirectory: ../output_docs/stavanger/research_statement
Gonzalez_RS.typ added for stavanger


## Teaching Statement

In [ ]:
rt_gen_df = pd.read_csv(path_output / "ts_docs_gpt-4.1-mini.csv")

gen_aux = to_generate_df[to_generate_df.need_TS == 1].copy()

generate_docs(gen_aux, rt_gen_df, gen_type="ts")

Processing uab_applied...
Main Directory already exists: ../output_docs/uab_applied
Creating subdirectory: ../output_docs/uab_applied/teaching_statement
Gonzalez_TS.typ added for uab_applied
